# LCED Cohort Study: T2D Second-Line Therapy

This notebook implements a complete cohort study using IBM LCED data, translated from the original R `lced_template_analysis.R`. The study examines **second-line antidiabetic therapy initiation** in patients with Type 2 Diabetes (T2D) who have poor glycaemic control on metformin.

## Study Design

| Step | Definition |
|---|---|
| **Eligible population** | Adults (≥ 45 years) with ≥ 1 T2D ICD code (ICD-9: 250.x; ICD-10: E11.x) |
| **Index date** | First metformin prescription date |
| **Poor glycaemic control** | HbA1c ≥ 7 % (LOINC 4548-4) after metformin start, with HbA1c ≥ 9 % **or** two measurements ≤ 183 days apart |
| **Exposure** | First second-line drug (GLP-1, SGLT-2, DPP-4, or sulfonylurea) within 365 days of poor-control HbA1c |
| **Outcomes (MACE)** | AMI, stroke, CHF — from `v_inpatient_services` after exposure date |
| **Follow-up** | Months of continuous enrollment after second-line start |

In [1]:
import numpy as np
import pandas as pd

## 1. Synthetic LCED Tables

Below we build synthetic DataFrames that mirror the LCED schema. Replace these with real `v_*` table pulls in a live environment.

In [2]:
np.random.seed(0)

N_PAT = 20
pids = [f"P{i:04d}" for i in range(1, N_PAT + 1)]

# --- Enrollment summary ---
v_annual = pd.DataFrame(
    {
        "patient_id": pids,
        "dobyr": np.random.randint(1940, 1975, N_PAT),
        "sex": np.random.choice(["M", "F"], N_PAT),
        **{f"enrind{m}": np.random.choice([0, 1], N_PAT, p=[0.05, 0.95]) for m in range(1, 13)},
        "year": [2015] * N_PAT,
    }
)

# --- Outpatient services (diagnosis) ---
_dx_pool = [
    ("E119", "0"),
    ("E118", "0"),
    ("25000", "9"),
    ("25002", "9"),
    ("I509", "0"),
    ("41001", "9"),
    ("I259", "0"),
    ("Z794", "0"),
]

rows = []
for pid in pids:
    for _ in range(np.random.randint(2, 6)):
        dx, dxver = _dx_pool[np.random.randint(len(_dx_pool))]
        rows.append(
            {
                "patient_id": pid,
                "svcdate": pd.Timestamp("2013-01-01") + pd.Timedelta(days=int(np.random.randint(0, 1000))),
                "dx1": dx,
                "dxver": dxver,
            }
        )
v_outpatient = pd.DataFrame(rows)

# --- Drug claims ---
# NDC codes: metformin, GLP-1s, SGLT-2s, DPP-4s, SUs  (synthetic 11-digit codes)
MET_NDCS = ["00093717401", "00093717501"]
GLP1_NDCS = ["00169368812", "00169368915"]  # exenatide, liraglutide
SGLT2_NDCS = ["00310653090", "57844011290"]  # dapagliflozin, canagliflozin
DPP4_NDCS = ["00006054054", "00310027190"]  # sitagliptin, saxagliptin
SU_NDCS = ["00093116401", "00093013901"]  # glimepiride, glyburide

SECOND_LINE = GLP1_NDCS + SGLT2_NDCS + DPP4_NDCS + SU_NDCS
DRUG_CLASS = (
    dict.fromkeys(GLP1_NDCS, "glp1")
    | dict.fromkeys(SGLT2_NDCS, "sglt2")
    | dict.fromkeys(DPP4_NDCS, "dpp4")
    | dict.fromkeys(SU_NDCS, "su")
)

drg_rows = []
for pid in np.random.choice(pids, 40, replace=True):
    pool = MET_NDCS + SECOND_LINE
    ndc = pool[np.random.randint(len(pool))]
    date = pd.Timestamp("2014-01-01") + pd.Timedelta(days=int(np.random.randint(0, 1200)))
    drg_rows.append({"patient_id": pid, "svcdate": date, "ndcnum": ndc, "daysupp": 30, "refill": 0})
v_drug_claims = pd.DataFrame(drg_rows)

# --- Observations (HbA1c) ---
HBA1C_LOINCS = ["4548-4", "17856-6"]
obs_rows = []
for pid in np.random.choice(pids, 60, replace=True):
    loinc = np.random.choice(HBA1C_LOINCS)
    value = round(np.random.uniform(5.5, 11.0), 1)
    date = pd.Timestamp("2014-06-01") + pd.Timedelta(days=int(np.random.randint(0, 900)))
    obs_rows.append(
        {"patient_id": pid, "observation_date": date, "loinc_test_id": loinc, "std_value": value, "std_uom": "%"}
    )
v_observation = pd.DataFrame(obs_rows)

# --- Inpatient services (MACE outcomes) ---
MACE_CODES = ["I219", "I639", "I509", "41001", "43401"]
inp_rows = []
for pid in np.random.choice(pids, 10, replace=True):
    dx = MACE_CODES[np.random.randint(len(MACE_CODES))]
    date = pd.Timestamp("2015-01-01") + pd.Timedelta(days=int(np.random.randint(0, 1000)))
    inp_rows.append({"patient_id": pid, "svcdate": date, "dx1": dx, "dx2": None})
v_inpatient_services = pd.DataFrame(inp_rows)

print(f"Enrollment rows     : {len(v_annual)}")
print(f"Outpatient dx rows  : {len(v_outpatient)}")
print(f"Drug claim rows     : {len(v_drug_claims)}")
print(f"Observation rows    : {len(v_observation)}")
print(f"Inpatient rows      : {len(v_inpatient_services)}")

Enrollment rows     : 20
Outpatient dx rows  : 62
Drug claim rows     : 40
Observation rows    : 60
Inpatient rows      : 10


## 2. Eligible Population

### Step 2a — T2D Cohort

Patients with ≥ 1 T2D code across any outpatient, inpatient, or facility table.

In [3]:
T2D_ICD9 = [
    "25000",
    "25002",
    "25010",
    "25012",
    "25040",
    "25042",
    "25050",
    "25052",
    "25060",
    "25062",
    "25070",
    "25072",
    "25090",
    "25092",
]


def is_t2d(dx_series):
    nodot = dx_series.str.replace(".", "", regex=False)
    icd9 = nodot.isin(T2D_ICD9)
    icd10 = dx_series.str.startswith("E11", na=False)
    return icd9 | icd10


t2d_flag = is_t2d(v_outpatient["dx1"])
t2d_patients = v_outpatient[t2d_flag]["patient_id"].unique()
print(f"Patients with ≥1 T2D code: {len(t2d_patients)}")

Patients with ≥1 T2D code: 17


### Step 2b — Age Filter (≥ 45 years at index)

In [4]:
# Compute age at 2014-01-01 (the study start year in the original R analysis)
STUDY_START = pd.Timestamp("2014-01-01")
enroll = v_annual[["patient_id", "dobyr"]].drop_duplicates()
enroll["age_at_study"] = STUDY_START.year - enroll["dobyr"]

old_patients = enroll[enroll["age_at_study"] >= 45]["patient_id"].values
eligible = np.intersect1d(t2d_patients, old_patients)
print(f"T2D patients aged ≥ 45: {len(eligible)}")

T2D patients aged ≥ 45: 17


## 3. Metformin Index Date

The index date is the **first** metformin claim date per patient.

In [5]:
met_claims = v_drug_claims[v_drug_claims["patient_id"].isin(eligible) & v_drug_claims["ndcnum"].isin(MET_NDCS)].copy()

met_index = met_claims.groupby("patient_id")["svcdate"].min().rename("metformin_start").reset_index()

print(f"Patients with metformin: {len(met_index)}")
met_index.head()

Patients with metformin: 4


,patient_id,metformin_start
0,P0001,2016-01-01
1,P0005,2016-11-25
2,P0007,2015-03-25
3,P0020,2016-04-28


## 4. Poor Glycaemic Control

Filter HbA1c observations to:
1. After metformin start
2. After 2014-01-01 (study window)
3. HbA1c ≥ 7 %

Then flag poor control: value ≥ 9 % **OR** two consecutive measurements ≤ 183 days apart.
Keep only the **first** qualifying observation per patient.

In [6]:
a1c = (
    v_observation[v_observation["loinc_test_id"].isin(HBA1C_LOINCS)]
    .copy()
    .rename(columns={"observation_date": "obs_date", "std_value": "hba1c"})[["patient_id", "obs_date", "hba1c"]]
)

# Merge with metformin index
a1c = a1c.merge(met_index, on="patient_id", how="inner")

# Apply filters: after metformin start, after study start, HbA1c ≥ 7
a1c = (
    a1c[(a1c["obs_date"] > a1c["metformin_start"]) & (a1c["obs_date"] > STUDY_START) & (a1c["hba1c"] >= 7.0)]
    .sort_values(["patient_id", "obs_date"])
    .reset_index(drop=True)
)

# Days between consecutive measurements per patient
a1c["prev_date"] = a1c.groupby("patient_id")["obs_date"].shift(1)
a1c["days_since_last"] = (a1c["obs_date"] - a1c["prev_date"]).dt.days

# Poor control flag
a1c["poor_control"] = (a1c["hba1c"] >= 9.0) | (a1c["days_since_last"] <= 183)

# First qualifying poor-control event per patient
poor_ctrl = (
    a1c[a1c["poor_control"]]
    .groupby("patient_id")
    .first()
    .reset_index()[["patient_id", "obs_date", "hba1c", "metformin_start"]]
    .rename(columns={"obs_date": "poor_ctrl_date"})
)

print(f"Patients with poor glycaemic control: {len(poor_ctrl)}")
poor_ctrl.head()

Patients with poor glycaemic control: 0


,patient_id,poor_ctrl_date,hba1c,metformin_start


## 5. Second-Line Therapy Assignment

For each patient in the poor-control cohort, find the **first** second-line drug claim that:
- Falls **after** the poor-control HbA1c date
- Falls **within 365 days** of the poor-control date
- Has a gap of > 180 days from any previous second-line claim (new initiation, not refill)

In [7]:
sl_claims = v_drug_claims[
    v_drug_claims["patient_id"].isin(poor_ctrl["patient_id"]) & v_drug_claims["ndcnum"].isin(SECOND_LINE)
].copy()
sl_claims["drug_class"] = sl_claims["ndcnum"].map(DRUG_CLASS)

# Merge with poor-control date
sl_claims = sl_claims.merge(poor_ctrl[["patient_id", "poor_ctrl_date"]], on="patient_id", how="inner")

# Filter: after poor-control date and within 365 days
sl_claims["days_after_ctrl"] = (sl_claims["svcdate"] - sl_claims["poor_ctrl_date"]).dt.days
sl_claims = (
    sl_claims[(sl_claims["days_after_ctrl"] >= 0) & (sl_claims["days_after_ctrl"] <= 365)]
    .sort_values(["patient_id", "svcdate"])
    .reset_index(drop=True)
)

# Keep first claim per patient (new initiation)
sl_claims["prev_sl_date"] = sl_claims.groupby("patient_id")["svcdate"].shift(1)
sl_claims["sl_gap"] = (sl_claims["svcdate"] - sl_claims["prev_sl_date"]).dt.days
sl_claims["is_new"] = sl_claims["prev_sl_date"].isna() | (sl_claims["sl_gap"] > 180)

cohort = (
    sl_claims[sl_claims["is_new"]]
    .groupby("patient_id")
    .first()
    .reset_index()[["patient_id", "svcdate", "drug_class", "poor_ctrl_date"]]
    .rename(columns={"svcdate": "second_line_start"})
)

print(f"Final cohort size: {len(cohort)}")
print()
print("Drug class distribution:")
print(cohort["drug_class"].value_counts().to_string())

Final cohort size: 0

Drug class distribution:
Series([], )


## 6. MACE Outcome Ascertainment

MACE (Major Adverse Cardiovascular Events) = AMI, Stroke, or CHF from inpatient services after second-line start.

In [8]:
AMI_ICD9_PFX = ["410"]
AMI_ICD10_PFX = ["I21", "I22"]
STROKE_ICD9_PFX = ["160", "161", "162", "163", "430", "431", "432", "433", "434", "435"]
STROKE_ICD10_PFX = ["I60", "I61", "I62", "I63", "I64"]
CHF_ICD9 = ["4280", "4281", "4282", "4283", "4284", "4289"]
CHF_ICD10_PFX = ["I50"]


def flag_outcome(dx_series, icd9_pfx=None, icd10_pfx=None, icd9_exact=None):
    m = pd.Series(False, index=dx_series.index)
    if icd9_pfx:
        for p in icd9_pfx:
            m |= dx_series.str.startswith(p, na=False)
    if icd10_pfx:
        for p in icd10_pfx:
            m |= dx_series.str.startswith(p, na=False)
    if icd9_exact:
        m |= dx_series.isin(icd9_exact)
    return m


inp = v_inpatient_services.copy()

for col in ["dx1", "dx2"]:
    if col not in inp.columns:
        inp[col] = pd.NA

inp["ami"] = flag_outcome(inp["dx1"], icd9_pfx=AMI_ICD9_PFX, icd10_pfx=AMI_ICD10_PFX)
inp["stroke"] = flag_outcome(inp["dx1"], icd9_pfx=STROKE_ICD9_PFX, icd10_pfx=STROKE_ICD10_PFX)
inp["chf"] = flag_outcome(inp["dx1"], icd9_exact=CHF_ICD9, icd10_pfx=CHF_ICD10_PFX)
inp["mace"] = inp["ami"] | inp["stroke"] | inp["chf"]

# Merge with cohort and keep events after second-line start
inp_cohort = inp[inp["mace"] & inp["patient_id"].isin(cohort["patient_id"])].merge(
    cohort[["patient_id", "second_line_start"]], on="patient_id", how="inner"
)
inp_cohort = inp_cohort[inp_cohort["svcdate"] > inp_cohort["second_line_start"]]

for outcome in ["ami", "stroke", "chf"]:
    n = inp_cohort[inp_cohort[outcome]]["patient_id"].nunique()
    print(f"{outcome.upper():6s}: {n} patients")

AMI   : 0 patients
STROKE: 0 patients
CHF   : 0 patients


## 7. Coverage / Follow-Up Calculation

Follow-up is defined as the number of continuous enrolled months after the second-line start date. Enrollment gaps are encoded as `enrind{1..12} == 0` in `v_annual_summary_enrollment`.

In [9]:
enrind_cols = [f"enrind{m}" for m in range(1, 13)]


def followup_months(pid, start_date, enrollment_df, enrind_cols):
    """Return months of continuous enrollment after start_date."""
    rows = enrollment_df[enrollment_df["patient_id"] == pid].sort_values("year")
    count = 0
    for _, row in rows.iterrows():
        for m_idx, col in enumerate(enrind_cols, start=1):
            month_date = pd.Timestamp(year=int(row["year"]), month=m_idx, day=1)
            if month_date < start_date:
                continue
            if row[col] == 0:
                return count
            count += 1
    return count


cohort_follow = cohort.copy()
cohort_follow["followup_months"] = cohort_follow.apply(
    lambda r: followup_months(r["patient_id"], r["second_line_start"], v_annual, enrind_cols),
    axis=1,
)

print("Follow-up summary (months after second-line start):")
print(cohort_follow["followup_months"].describe().round(1))
cohort_follow[["patient_id", "drug_class", "second_line_start", "followup_months"]].head(8)

Follow-up summary (months after second-line start):
count    0.0
mean     NaN
std      NaN
min      NaN
25%      NaN
50%      NaN
75%      NaN
max      NaN
Name: followup_months, dtype: float64


,patient_id,drug_class,second_line_start,followup_months


## 8. EHRData Bridge

Convert the final cohort to an `EHRData` object for downstream ML / survival analysis.

In [10]:
from ehrdata.io.source import to_ehrdata

# Build canonical tables for cohort patients only
cohort_ids = set(cohort["patient_id"])

# patinfo from enrollment
patinfo = (
    v_annual[["patient_id", "dobyr", "sex"]]
    .drop_duplicates(subset="patient_id")
    .query("patient_id in @cohort_ids")
    .reset_index(drop=True)
)

# diagnosis canonical table
diag_raw = v_outpatient[["patient_id", "svcdate", "dx1", "dxver"]].copy()
diag_raw = diag_raw[diag_raw["patient_id"].isin(cohort_ids)]
diag_raw = diag_raw.rename(columns={"svcdate": "eventdate", "dx1": "dx"})
diag_raw["dxver"] = diag_raw["dxver"].fillna(diag_raw["dx"].apply(lambda d: "9" if d and d[0].isdigit() else "0"))

# therapy canonical table from drug claims
therapy_raw = (
    v_drug_claims[v_drug_claims["patient_id"].isin(cohort_ids)]
    .rename(columns={"svcdate": "fill_date", "ndcnum": "ndc11"})
    .assign(
        ingredient=lambda df: df["ndc11"].map(DRUG_CLASS).fillna("metformin"),
        rxcui=None,
        prescription_date=pd.NaT,
        start_date=pd.NaT,
        end_date=pd.NaT,
        refill=pd.array([pd.NA] * len(v_drug_claims[v_drug_claims["patient_id"].isin(cohort_ids)]), dtype="Int64"),
    )[
        [
            "patient_id",
            "fill_date",
            "ingredient",
            "ndc11",
            "rxcui",
            "prescription_date",
            "start_date",
            "end_date",
            "refill",
        ]
    ]
)

edata = to_ehrdata(
    patinfo,
    diagnosis=diag_raw,
    therapy=therapy_raw,
    source="lced",
)
print(edata)

EHRData object with n_obs × n_vars = 0 × 0
    obs: 'dobyr', 'sex'
    var: 'concept_source', 'concept_code'
    uns: 'source_io_source', 'source_io_tables'
    shape of .X: (0, 0)


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/functools.py:909: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)


In [11]:
# Attach cohort metadata to obs
cohort_meta = cohort_follow[["patient_id", "drug_class", "second_line_start", "followup_months"]].set_index(
    "patient_id"
)
# Align to edata.obs index
common = edata.obs_names.intersection(cohort_meta.index)
edata.obs.loc[common, "drug_class"] = cohort_meta.loc[common, "drug_class"]
edata.obs.loc[common, "second_line_start"] = cohort_meta.loc[common, "second_line_start"]
edata.obs.loc[common, "followup_months"] = cohort_meta.loc[common, "followup_months"]

print("obs columns:", edata.obs.columns.tolist())
print()
print(edata.obs.head(5).to_string())

obs columns: ['dobyr', 'sex', 'drug_class', 'second_line_start', 'followup_months']

Empty DataFrame
Columns: [dobyr, sex, drug_class, second_line_start, followup_months]
Index: []


## Summary

| Cohort step | Key operation |
|---|---|
| T2D definition | `dx.str.startswith("E11")` \| `dx.isin(T2D_ICD9)` |
| Age filter | enrollment `dobyr` vs study year |
| Metformin index date | `groupby("patient_id")["svcdate"].min()` |
| Poor control HbA1c | `groupby` + `shift` + `diff` on sorted dates |
| Second-line assignment | NDC lookup + 365-day window + 180-day gap filter |
| MACE outcomes | `str.startswith` on ICD prefix lists |
| Follow-up | `enrind{1..12}` monthly enrollment scan |
| EHRData export | `to_ehrdata(patinfo, diagnosis=..., therapy=...)` |

**Next steps**: Attach MACE outcomes as time-to-event columns in `edata.obs` and fit a survival model using `lifelines` or `scikit-survival`.